# Agentic Customer Churn Retention System
**Pipeline:** XGBoost Prediction → SHAP Explanation → LangGraph Multi-Agent → LLM Planner (Groq) → CRM + Slack Tools

---

## STEP 0 — Install Dependencies

In [ ]:
!pip install langgraph langchain xgboost shap openpyxl slack_sdk --quiet

## STEP 1 — Configuration (Set Your Keys Here)

In [ ]:
# ============================================================
# CONFIGURATION — edit these values before running
# ============================================================

GROQ_API_KEY    = ""       # get free key at console.groq.com
SLACK_BOT_TOKEN = "" # enter slack token
SLACK_CHANNEL   = "launches"
GROQ_MODEL      = "llama-3.1-8b-instant"
GROQ_URL        = "https://api.groq.com/openai/v1/chat/completions"

print("Configuration loaded.")

Configuration loaded.


## STEP 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import requests
import os
from datetime import datetime
from typing import TypedDict, List, Optional

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier
import shap

from langgraph.graph import StateGraph, END
from slack_sdk import WebClient # New import for Slack bot token

print("All imports successful.")

All imports successful.


## STEP 3 — Load and Prepare Dataset

In [ ]:
# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_excel("Telco_customer_churn.xlsx")
print("Initial Shape:", df.shape)

# ----------------------------
# Clean column names
# ----------------------------
df.columns = df.columns.str.strip()

# ----------------------------
# Fix Total Charges
# ----------------------------
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["Total Charges"] = df["Total Charges"].fillna(df["Total Charges"].median())

# ----------------------------
# Drop leakage + useless columns
# ----------------------------
drop_cols = [
    "CustomerID", "Count",
    "Country", "State", "City", "Zip Code",
    "Lat Long", "Latitude", "Longitude",
    "Churn Label", "Churn Score", "CLTV", "Churn Reason"
]
df.drop(columns=drop_cols, inplace=True, errors="ignore")

# ----------------------------
# Target and features
# ----------------------------
y = df["Churn Value"].astype(int)
X = df.drop("Churn Value", axis=1)
X = pd.get_dummies(X, drop_first=True)

print("Processed Shape:", X.shape)

Initial Shape: (7043, 33)
Processed Shape: (7043, 30)


## STEP 4 — Train / Evaluate XGBoost Model

In [ ]:
# ----------------------------
# Train / test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

# ----------------------------
# Train model
# ----------------------------
model = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=2.5   # improves recall for churn class
)
model.fit(X_train, y_train)
print("Model training complete.")

# ----------------------------
# Save model
# ----------------------------
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Model saved as model.pkl")

# ----------------------------
# Evaluation
# ----------------------------
y_pred = model.predict(X_test)

print("\n===== Evaluation Metrics =====")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Train: (5634, 30) | Test: (1409, 30)
Model training complete.
Model saved as model.pkl

===== Evaluation Metrics =====
Accuracy : 0.7708
Precision: 0.5556
Recall   : 0.6818
F1 Score : 0.6122

Confusion Matrix:
[[831 204]
 [119 255]]


## STEP 5 — SHAP Explainability

In [ ]:
# ----------------------------
# Build SHAP explainer
# ----------------------------
shap_explainer = shap.TreeExplainer(model)

# Keep X_test accessible globally for the agent nodes
X_TEST_GLOBAL = X_test.reset_index(drop=True)


def get_shap_reasons(customer_index: int, top_k: int = 3):
    """
    Returns (shap_reasons: List[str], raw_shap_values: List[float])
    for a given customer index in X_TEST_GLOBAL.
    """
    row = X_TEST_GLOBAL.iloc[customer_index].values.reshape(1, -1)
    shap_values = shap_explainer.shap_values(row)

    # Handle both list and array output
    if isinstance(shap_values, list):
        sv = shap_values[1][0]
    else:
        sv = shap_values[0]

    features = X_TEST_GLOBAL.columns
    paired = sorted(zip(features, sv), key=lambda x: abs(x[1]), reverse=True)[:top_k]

    reasons = []
    raw_vals = []
    for feat, val in paired:
        direction = "increases" if val > 0 else "reduces"
        strength  = "strongly" if abs(val) > 2 else ("moderately" if abs(val) > 1 else "slightly")
        reasons.append(f"{feat} = {X_TEST_GLOBAL.iloc[customer_index][feat]} {strength} {direction} churn risk (SHAP={round(val,3)})")
        raw_vals.append(float(val))

    return reasons, raw_vals


# Quick smoke test
r, v = get_shap_reasons(0)
print("Sample SHAP reasons for customer 0:")
for reason in r:
    print(" -", reason)

Sample SHAP reasons for customer 0:
 - Contract_Two year = True strongly reduces churn risk (SHAP=-2.742000102996826)
 - Tenure Months = 72 strongly reduces churn risk (SHAP=-2.013000011444092)
 - Total Charges = 8468.2 slightly increases churn risk (SHAP=0.8930000066757202)


## STEP 6 — Tool Functions (CRM, Discount, Slack)

In [ ]:
CRM_FILE   = "customer_state.json"
SLACK_LOG  = "slack_log.json"

# Initialize Slack client globally if token is present
slack_client = WebClient(token=SLACK_BOT_TOKEN) if SLACK_BOT_TOKEN else None


# ──────────────────────────────
# TOOL 1: CRM Update
# ──────────────────────────────
def update_crm(customer_id: str, field: str, value) -> dict:
    """Persist a customer field update to the local CRM JSON file."""
    try:
        with open(CRM_FILE, "r") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        data = {}

    if customer_id not in data:
        data[customer_id] = {}

    data[customer_id][field]          = value
    data[customer_id]["last_updated"] = datetime.now().isoformat()

    with open(CRM_FILE, "w") as f:
        json.dump(data, f, indent=4)

    print(f"[CRM] Customer {customer_id}: {field} → {value}")
    return data[customer_id]


# ──────────────────────────────
# TOOL 2: Discount Calculator
# ──────────────────────────────
def calculate_discount(churn_probability: float, customer_ltv: float = 1500.0) -> dict:
    """
    Calculate a retention discount based on churn risk and LTV.
    Returns a discount between 10% and 30%.
    """
    raw      = churn_probability * customer_ltv / 50 # Changed divisor from 1000 to 50
    discount = round(max(10.0, min(30.0, raw)), 2)

    result = {
        "ltv": customer_ltv,
        "churn_probability": churn_probability,
        "discount_percentage": discount,
        "timestamp": datetime.now().isoformat()
    }
    print(f"[Discount] {discount}% discount calculated for LTV={customer_ltv}, risk={churn_probability}")
    return result


# ──────────────────────────────
# TOOL 3: Slack Notification (using Bot Token)
# ──────────────────────────────
def send_slack_alert(message: str, channel: str = SLACK_CHANNEL) -> dict:
    """
    Send a Slack alert via Bot Token. Skips gracefully if token not configured.
    Always logs to slack_log.json regardless.
    """
    status = "skipped_no_bot_token"

    if slack_client:
        try:
            response = slack_client.chat_postMessage(channel=channel, text=message)
            status = "sent" if response["ok"] else f"failed_{response['error']}"
        except Exception as e:
            status = f"error_{str(e)[:40]}"

    log_entry = {
        "message":   message,
        "channel":   channel,
        "status":    status,
        "timestamp": datetime.now().isoformat()
    }

    # Append to log file
    try:
        with open(SLACK_LOG, "r") as f:
            logs = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        logs = []
    logs.append(log_entry)
    with open(SLACK_LOG, "w") as f:
        json.dump(logs, f, indent=4)

    print(f"[Slack] {status}: {message[:80]}")
    return log_entry


print("Tool functions defined: update_crm, calculate_discount, send_slack_alert")

Tool functions defined: update_crm, calculate_discount, send_slack_alert


## STEP 7 — Groq LLM Planner

In [ ]:
ALLOWED_ACTIONS = ["send_email", "offer_discount", "schedule_call", "escalate_human", "do_nothing"]


def compute_confidence(raw_shap_values: list) -> str:
    """Convert SHAP magnitudes into a confidence level."""
    if not raw_shap_values:
        return "low"
    top = max(abs(v) for v in raw_shap_values)
    if top > 0.3:
        return "high"
    elif top < 0.15:
        return "low"
    return "medium"


def rule_based_override(churn_probability: float, raw_shap_values: list):
    """
    Safety layer — handles edge cases deterministically so the LLM
    is only invoked for the ambiguous middle range.
    Returns a decision dict or None (meaning: let the LLM decide).
    """
    top = max((abs(v) for v in raw_shap_values), default=0)

    if churn_probability < 0.6:
        return {"action": "do_nothing",       "reasoning": "Low churn probability — no intervention needed."}
    if churn_probability > 0.9:
        return {"action": "offer_discount",   "reasoning": "Extremely high churn risk — immediate discount applied."}
    if churn_probability > 0.6 and top < 0.15:
        return {"action": "escalate_human",   "reasoning": "Medium risk but low SHAP confidence — human review needed."}

    return None  # LLM decides


def build_planner_prompt(shap_reasons: list, confidence: str) -> str:
    reasons_text = "\n".join(f"{i+1}. {r}" for i, r in enumerate(shap_reasons))
    return f"""
You are a strict decision-making engine for telecom customer retention.

====================
INPUT INFORMATION
====================
SHAP Reasons (ranked by importance):
{reasons_text}

Confidence Level: {confidence}

====================
ALLOWED ACTIONS
====================
- send_email
- offer_discount
- schedule_call
- escalate_human
- do_nothing

====================
DECISION RULES
====================
- Use SHAP reasons as the primary signal for your decision.
- High confidence + high churn signals  → offer_discount or schedule_call
- Medium confidence                     → send_email or schedule_call
- Low confidence                        → escalate_human or do_nothing
- Be consistent and deterministic.
- Reasoning must mention the top 1–2 SHAP factors and explain the cause → action link.
- Keep reasoning to ONE concise sentence.

====================
OUTPUT FORMAT (STRICT)
====================
Return ONLY valid JSON — no markdown, no explanation outside JSON:
{{"action": "one_of_allowed_actions", "reasoning": "one concise sentence"}}
"""


def call_groq_llm(prompt: str) -> dict:
    """Call the Groq LLM and return a parsed action/reasoning dict."""
    payload = {
        "model": GROQ_MODEL,
        "messages": [
            {"role": "system", "content": "Return ONLY valid JSON. No markdown, no preamble."},
            {"role": "user",   "content": prompt}
        ],
        "temperature": 0.2,
        "max_tokens": 200
    }
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(GROQ_URL, headers=headers, json=payload, timeout=10)
        data     = response.json()

        if "choices" not in data:
            return {"action": "escalate_human", "reasoning": f"LLM API error: {data}"}

        content = data["choices"][0]["message"]["content"]
        content = content.strip().replace("```json", "").replace("```", "")
        parsed  = json.loads(content)

        # Validate action is one of allowed values
        if parsed.get("action") not in ALLOWED_ACTIONS:
            parsed["action"] = "escalate_human"
            parsed["reasoning"] += " [action corrected by safety check]"

        return parsed

    except json.JSONDecodeError:
        return {"action": "escalate_human", "reasoning": "LLM returned invalid JSON."}
    except Exception as e:
        return {"action": "escalate_human", "reasoning": f"System error: {str(e)[:80]}"}


def llm_planner(churn_probability: float, shap_reasons: list, raw_shap_values: list) -> dict:
    """
    Full planner logic:
      1. Rule-based override for edge cases
      2. LLM decision for ambiguous middle range
    Returns {action, reasoning, source} dict.
    """
    override = rule_based_override(churn_probability, raw_shap_values)
    if override is not None:
        override["source"] = "rule"
        print(f"[Planner] Rule override → {override['action']}")
        return override

    confidence = compute_confidence(raw_shap_values)
    prompt     = build_planner_prompt(shap_reasons, confidence)
    decision   = call_groq_llm(prompt)
    decision["source"] = "llm"
    print(f"[Planner] LLM decision → {decision['action']} | {decision['reasoning']}")
    return decision


print("LLM planner functions defined.")

LLM planner functions defined.


## STEP 8 — LangGraph Agent State

In [ ]:
class ChurnState(TypedDict):
    # Input
    customer_id:        str
    customer_index:     int             # index into X_TEST_GLOBAL
    customer_ltv:       float           # lifetime value for discount calc

    # Produced by Monitor
    churn_probability:  float
    risk_level:         str             # "high" | "medium" | "low"

    # Produced by Explainer
    shap_reasons:       List[str]
    raw_shap_values:    List[float]

    # Produced by Planner (LLM)
    action:             str
    planner_reasoning:  str
    decision_source:    str             # "rule" | "llm"

    # Produced by Executor (tools)
    discount_info:      Optional[dict]
    crm_record:         Optional[dict]
    slack_log:          Optional[dict]
    execution_summary:  str

    # Produced by Negotiator
    customer_response:  str
    negotiation_round:  int

    # Metadata
    conversation_history: List[str]
    timestamps:           dict

print("ChurnState schema defined.")

ChurnState schema defined.


## STEP 9 — Agent Node Functions

In [ ]:
# ──────────────────────────────────────────────────────────────
# NODE 1: MONITOR
# Classifies the churn probability into a risk level.
# ──────────────────────────────────────────────────────────────
def node_monitor(state: ChurnState) -> ChurnState:
    print("\n[Monitor] Evaluating churn risk...")
    prob = state["churn_probability"]

    if prob >= 0.7:
        risk = "high"
    elif prob >= 0.4:
        risk = "medium"
    else:
        risk = "low"

    state["risk_level"] = risk
    state["timestamps"]["monitor"] = datetime.now().isoformat()
    state["conversation_history"].append(f"Monitor: churn_prob={prob:.2f}, risk={risk}")

    print(f"[Monitor] Churn probability={prob:.2f} → Risk level={risk}")
    return state

In [ ]:
# ──────────────────────────────────────────────────────────────
# NODE 2: EXPLAINER
# Runs SHAP on the actual customer row to generate reasons.
# ──────────────────────────────────────────────────────────────
def node_explainer(state: ChurnState) -> ChurnState:
    print("\n[Explainer] Computing SHAP values...")
    idx = state["customer_index"]

    reasons, raw_vals = get_shap_reasons(idx, top_k=3)

    state["shap_reasons"]    = reasons
    state["raw_shap_values"] = raw_vals
    state["timestamps"]["explainer"] = datetime.now().isoformat()
    state["conversation_history"].append(f"Explainer: {len(reasons)} SHAP reasons extracted")

    print(f"[Explainer] Top reasons:")
    for r in reasons:
        print(f"  - {r}")
    return state

In [ ]:
# ──────────────────────────────────────────────────────────────
# NODE 3: PLANNER  (LLM-powered)
# Uses Groq LLM (with rule-based safety override) to decide
# the best retention action based on SHAP evidence.
# ──────────────────────────────────────────────────────────────
def node_planner(state: ChurnState) -> ChurnState:
    print("\n[Planner] Calling LLM for retention decision...")

    decision = llm_planner(
        churn_probability = state["churn_probability"],
        shap_reasons      = state["shap_reasons"],
        raw_shap_values   = state["raw_shap_values"]
    )

    state["action"]           = decision["action"]
    state["planner_reasoning"] = decision["reasoning"]
    state["decision_source"]  = decision.get("source", "unknown")
    state["timestamps"]["planner"] = datetime.now().isoformat()
    state["conversation_history"].append(
        f"Planner ({decision.get('source','?')}): action={decision['action']}"
    )

    return state

In [ ]:
# ──────────────────────────────────────────────────────────────
# NODE 4: EXECUTOR
# Executes the planned action using real tool calls:
#   - CRM update
#   - Discount calculation (if applicable)
#   - Slack notification
# ──────────────────────────────────────────────────────────────
def node_executor(state: ChurnState) -> ChurnState:
    print("\n[Executor] Executing action with tools...")
    action      = state["action"]
    customer_id = state["customer_id"]
    prob        = state["churn_probability"]
    ltv         = state.get("customer_ltv", 1500.0)

    discount_info = None
    summary_parts = []

    # ── 1. Always update CRM ──
    crm = update_crm(customer_id, "last_action", action)
    update_crm(customer_id, "churn_probability", round(prob, 4))
    update_crm(customer_id, "risk_level", state["risk_level"])
    summary_parts.append(f"CRM updated: action={action}")

    # ── 2. Action-specific logic ──
    if action == "offer_discount":
        discount_info = calculate_discount(prob, ltv)
        update_crm(customer_id, "discount_offered", discount_info["discount_percentage"])
        summary_parts.append(f"Discount calculated: {discount_info['discount_percentage']}%")

    elif action == "send_email":
        summary_parts.append("Email queued for customer")

    elif action == "schedule_call":
        summary_parts.append("Call scheduled with retention team")

    elif action == "escalate_human":
        summary_parts.append("Escalated to human agent")

    elif action == "do_nothing":
        summary_parts.append("No intervention — customer is low risk")

    # ── 3. Slack alert ──
    slack_msg = (
        f"Churn Alert | Customer {customer_id} | "
        f"Risk={state['risk_level'].upper()} ({prob:.0%}) | "
        f"Action={action}"
    )
    if discount_info:
        slack_msg += f" | Discount={discount_info['discount_percentage']}%"
    slack_result = send_slack_alert(slack_msg)
    summary_parts.append(f"Slack: {slack_result['status']}")

    # ── 4. Update state ──
    state["discount_info"]      = discount_info
    state["crm_record"]         = crm
    state["slack_log"]          = slack_result
    state["execution_summary"]  = " | ".join(summary_parts)
    state["timestamps"]["executor"] = datetime.now().isoformat()
    state["conversation_history"].append(f"Executor: {state['execution_summary']}")

    print(f"[Executor] Done: {state['execution_summary']}")
    return state

In [ ]:
# ──────────────────────────────────────────────────────────────
# NODE 5: NEGOTIATOR
# Simulates customer response based on action and risk level.
# In a real system this would poll a CRM response feed.
# ──────────────────────────────────────────────────────────────
def node_negotiator(state: ChurnState) -> ChurnState:
    print("\n[Negotiator] Evaluating customer response...")
    action = state["action"]
    risk   = state["risk_level"]

    # Simulate response probabilities by action and risk
    response_map = {
        "offer_discount": {"high": "accepted",        "medium": "under_review", "low": "declined"},
        "send_email":     {"high": "opened_email",    "medium": "no_response",  "low": "no_response"},
        "schedule_call":  {"high": "call_confirmed",  "medium": "rescheduled",  "low": "no_response"},
        "escalate_human": {"high": "human_assigned",  "medium": "human_assigned","low": "closed"},
        "do_nothing":     {"high": "monitoring",      "medium": "monitoring",   "low": "monitoring"},
    }

    response = response_map.get(action, {}).get(risk, "no_response")

    state["customer_response"]   = response
    state["negotiation_round"]   = state.get("negotiation_round", 0) + 1
    state["timestamps"]["negotiator"] = datetime.now().isoformat()
    state["conversation_history"].append(f"Negotiator: customer_response={response}")

    print(f"[Negotiator] Customer response: {response}")
    return state

In [ ]:
# ──────────────────────────────────────────────────────────────
# NODE 6: MEMORY
# Persists the full final state to disk as agent memory.
# ──────────────────────────────────────────────────────────────
def node_memory(state: ChurnState) -> ChurnState:
    print("\n[Memory] Saving agent state to disk...")
    state["timestamps"]["memory"] = datetime.now().isoformat()

    # Make state JSON-serializable (handle any edge cases)
    def make_serializable(obj):
        if isinstance(obj, dict):
            return {k: make_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [make_serializable(i) for i in obj]
        elif isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj

    with open("agent_memory.json", "w") as f:
        json.dump(make_serializable(dict(state)), f, indent=4)

    print("[Memory] Saved to agent_memory.json")
    return state


print("All agent node functions defined.")

All agent node functions defined.


## STEP 10 — Build and Compile LangGraph

In [ ]:
# ──────────────────────────────────────────────────────────────
# Build the graph
# ──────────────────────────────────────────────────────────────
workflow = StateGraph(ChurnState)

# Register nodes
workflow.add_node("monitor",    node_monitor)
workflow.add_node("explainer",  node_explainer)
workflow.add_node("planner",    node_planner)
workflow.add_node("executor",   node_executor)
workflow.add_node("negotiator", node_negotiator)
workflow.add_node("memory",     node_memory)

# Entry point
workflow.set_entry_point("monitor")

# Edges — linear pipeline
workflow.add_edge("monitor",    "explainer")
workflow.add_edge("explainer",  "planner")
workflow.add_edge("planner",    "executor")
workflow.add_edge("executor",   "negotiator")
workflow.add_edge("negotiator", "memory")
workflow.add_edge("memory",     END)

# Compile
app = workflow.compile()
print("LangGraph compiled successfully.")

LangGraph compiled successfully.


## STEP 11 — Run the System on a Single Customer

In [ ]:
# ──────────────────────────────────────────────────────────────
# Run for a single customer
# Change customer_index to test different customers (0 to len(X_test)-1)
# ──────────────────────────────────────────────────────────────
CUSTOMER_INDEX = 5   # change this to test different customers

# Get the model's churn probability for this customer
row              = X_TEST_GLOBAL.iloc[CUSTOMER_INDEX].values.reshape(1, -1)
churn_prob       = float(model.predict_proba(row)[0][1])
actual_label     = int(y_test.iloc[CUSTOMER_INDEX])

print(f"Customer index : {CUSTOMER_INDEX}")
print(f"Churn prob     : {churn_prob:.4f}")
print(f"Actual label   : {'CHURN' if actual_label == 1 else 'NO CHURN'}")
print("\n" + "="*60)

# Build initial state
initial_state = {
    "customer_id":         f"cust_{CUSTOMER_INDEX:04d}",
    "customer_index":      CUSTOMER_INDEX,
    "customer_ltv":        1800.0,

    "churn_probability":   churn_prob,
    "risk_level":          "",

    "shap_reasons":        [],
    "raw_shap_values":     [],

    "action":              "",
    "planner_reasoning":   "",
    "decision_source":     "",

    "discount_info":       None,
    "crm_record":          None,
    "slack_log":           None,
    "execution_summary":   "",

    "customer_response":   "",
    "negotiation_round":   0,

    "conversation_history": [],
    "timestamps":           {}
}

# Invoke the graph
final_state = app.invoke(initial_state)

# ──────────────────────────────────────────────────────────────
# Pretty summary
# ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL SYSTEM SUMMARY")
print("="*60)
print(f"Customer ID      : {final_state['customer_id']}")
print(f"Churn Probability: {final_state['churn_probability']:.4f}")
print(f"Risk Level       : {final_state['risk_level'].upper()}")
print(f"SHAP Reasons     :")
for r in final_state["shap_reasons"]:
    print(f"  - {r}")
print(f"Action Decided   : {final_state['action']}")
print(f"Decision Source  : {final_state['decision_source']}")
print(f"LLM Reasoning    : {final_state['planner_reasoning']}")
if final_state["discount_info"]:
    print(f"Discount Offered : {final_state['discount_info']['discount_percentage']}%")
print(f"Customer Response: {final_state['customer_response']}")
print(f"Slack Status     : {final_state['slack_log']['status']}")
print("="*60)

Customer index : 5
Churn prob     : 0.7269
Actual label   : NO CHURN


[Monitor] Evaluating churn risk...
[Monitor] Churn probability=0.73 → Risk level=high

[Explainer] Computing SHAP values...
[Explainer] Top reasons:
  - Contract_Two year = False slightly increases churn risk (SHAP=0.4740000069141388)
  - Internet Service_Fiber optic = True slightly increases churn risk (SHAP=0.36000001430511475)
  - Total Charges = 1992.55 slightly reduces churn risk (SHAP=-0.3149999976158142)

[Planner] Calling LLM for retention decision...
[Planner] LLM decision → offer_discount | High confidence and significant SHAP factors (Contract_Two year and Internet Service_Fiber optic) indicate a high churn risk, warranting an offer to retain the customer.

[Executor] Executing action with tools...
[CRM] Customer cust_0005: last_action → offer_discount
[CRM] Customer cust_0005: churn_probability → 0.7269
[CRM] Customer cust_0005: risk_level → high
[Discount] 26.17% discount calculated for LTV=1800.0, risk

## STEP 12 — Batch Evaluation on Multiple Customers

In [ ]:
# ──────────────────────────────────────────────────────────────
# Run batch evaluation on a sample of customers
# and collect metrics across the pipeline
# ──────────────────────────────────────────────────────────────
import random

BATCH_SIZE     = 20    # increase for more thorough evaluation
RANDOM_SEED    = 42
random.seed(RANDOM_SEED)

# Sample customer indices
sample_indices = random.sample(range(len(X_TEST_GLOBAL)), BATCH_SIZE)

results = []

for idx in sample_indices:
    row        = X_TEST_GLOBAL.iloc[idx].values.reshape(1, -1)
    churn_prob = float(model.predict_proba(row)[0][1])
    actual     = int(y_test.iloc[idx])

    state = {
        "customer_id":          f"cust_{idx:04d}",
        "customer_index":       idx,
        "customer_ltv":         1500.0,
        "churn_probability":    churn_prob,
        "risk_level":           "",
        "shap_reasons":         [],
        "raw_shap_values":      [],
        "action":               "",
        "planner_reasoning":    "",
        "decision_source":      "",
        "discount_info":        None,
        "crm_record":           None,
        "slack_log":            None,
        "execution_summary":    "",
        "customer_response":    "",
        "negotiation_round":    0,
        "conversation_history": [],
        "timestamps":           {}
    }

    final = app.invoke(state)

    results.append({
        "customer_id":      final["customer_id"],
        "actual_churn":     actual,
        "churn_prob":       round(churn_prob, 4),
        "risk_level":       final["risk_level"],
        "action":           final["action"],
        "decision_source":  final["decision_source"],
        "customer_response":final["customer_response"],
        "discount_%":       final["discount_info"]["discount_percentage"] if final["discount_info"] else None
    })


# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("batch_results.csv", index=False)

print("\n===== BATCH EVALUATION RESULTS =====")
print(results_df.to_string(index=False))


[Monitor] Evaluating churn risk...
[Monitor] Churn probability=0.82 → Risk level=high

[Explainer] Computing SHAP values...
[Explainer] Top reasons:
  - Total Charges = 5427.05 moderately increases churn risk (SHAP=1.1349999904632568)
  - Tenure Months = 57 slightly reduces churn risk (SHAP=-0.9580000042915344)
  - Dependents_Yes = False slightly increases churn risk (SHAP=0.34299999475479126)

[Planner] Calling LLM for retention decision...
[Planner] LLM decision → offer_discount | High confidence and moderately increasing total charges (SHAP=1.1349999904632568) justify offering a discount to retain the customer.

[Executor] Executing action with tools...
[CRM] Customer cust_1309: last_action → offer_discount
[CRM] Customer cust_1309: churn_probability → 0.8235
[CRM] Customer cust_1309: risk_level → high
[Discount] 24.7% discount calculated for LTV=1500.0, risk=0.8234943747520447
[CRM] Customer cust_1309: discount_offered → 24.7
[Slack] sent: Churn Alert | Customer cust_1309 | Risk=H

In [ ]:
# ──────────────────────────────────────────────────────────────
# Compute agent-level evaluation metrics
# ──────────────────────────────────────────────────────────────
print("\n===== AGENT EVALUATION METRICS =====")

# 1. Action distribution
print("\n[Action Distribution]")
print(results_df["action"].value_counts().to_string())

# 2. Decision source (rule vs llm)
print("\n[Decision Source]")
print(results_df["decision_source"].value_counts().to_string())

# 3. Risk level distribution
print("\n[Risk Level Distribution]")
print(results_df["risk_level"].value_counts().to_string())

# 4. Intervention rate on actual churners
churners = results_df[results_df["actual_churn"] == 1]
intervened = churners[churners["action"] != "do_nothing"]
intervention_rate = len(intervened) / len(churners) if len(churners) > 0 else 0
print(f"\n[Intervention Rate on Actual Churners]: {intervention_rate:.2%} ({len(intervened)}/{len(churners)})")

# 5. Average discount for discount actions
discount_rows = results_df[results_df["discount_%"].notna()]
if len(discount_rows) > 0:
    avg_discount = discount_rows["discount_%"].mean()
    print(f"[Average Discount Offered]: {avg_discount:.2f}%")

# 6. Customer acceptance rate
accepted = results_df[results_df["customer_response"] == "accepted"]
print(f"[Customer Acceptance Rate]: {len(accepted)/len(results_df):.2%} ({len(accepted)}/{len(results_df)})")

print("\nBatch results saved to batch_results.csv")


===== AGENT EVALUATION METRICS =====

[Action Distribution]
action
do_nothing        11
offer_discount     8
send_email         1

[Decision Source]
decision_source
rule    14
llm      6

[Risk Level Distribution]
risk_level
high      9
low       9
medium    2

[Intervention Rate on Actual Churners]: 77.78% (7/9)
[Average Discount Offered]: 25.66%
[Customer Acceptance Rate]: 40.00% (8/20)

Batch results saved to batch_results.csv


In [ ]:
# ──────────────────────────────────────────────────────────────
# Compute additional agent-level evaluation metrics
# ──────────────────────────────────────────────────────────────
print("\n===== ADDITIONAL AGENT EVALUATION METRICS =====")

# 7. False Positive Rate (non-churners who got a discount)
false_positives = results_df[(results_df["actual_churn"] == 0) & (results_df["action"] == "offer_discount")]
fp_rate = len(false_positives) / len(results_df[results_df["actual_churn"] == 0]) if len(results_df[results_df["actual_churn"] == 0]) > 0 else 0
print(f"\n[False Positive Rate (Non-churners discounted)]: {fp_rate:.2%} ({len(false_positives)}/{len(results_df[results_df["actual_churn"] == 0])})")

# 8. Cost Analysis
# Assuming average LTV for customers is represented by `customer_ltv` in initial state, here used 1500.0
AVG_LTV = 1500.0

# Cost of unnecessary discounts (False Positives)
# For simplicity, assuming discount_percentage directly correlates to cost reduction from LTV
# and that these customers would not have churned anyway
if not false_positives.empty:
    cost_fp = (false_positives["discount_%"] / 100 * AVG_LTV).sum()
else:
    cost_fp = 0
print(f"[Estimated Cost of Unnecessary Discounts (False Positives)]: ${cost_fp:.2f}")

# Churn revenue saved (Actual Churners who got a discount and accepted)
# Assuming accepted discount means retained customer, saving their LTV
retained_churners = results_df[(results_df["actual_churn"] == 1) & (results_df["action"] == "offer_discount") & (results_df["customer_response"] == "accepted")]
revenue_saved = len(retained_churners) * AVG_LTV
print(f"[Estimated Churn Revenue Saved (Retained Churners)]: ${revenue_saved:.2f}")

print("\nUpdated batch evaluation metrics printed.")


===== ADDITIONAL AGENT EVALUATION METRICS =====

[False Positive Rate (Non-churners discounted)]: 18.18% (2/11)
[Estimated Cost of Unnecessary Discounts (False Positives)]: $675.45
[Estimated Churn Revenue Saved (Retained Churners)]: $9000.00

Updated batch evaluation metrics printed.


In [ ]:
# ──────────────────────────────────────────────────────────────
# Run batch evaluation on a sample of customers
# and collect metrics across the pipeline
# ──────────────────────────────────────────────────────────────
import random

BATCH_SIZE     = 20    # increase for more thorough evaluation
RANDOM_SEED    = 42
random.seed(RANDOM_SEED)

# Sample customer indices
sample_indices = random.sample(range(len(X_TEST_GLOBAL)), BATCH_SIZE)

results = []

for idx in sample_indices:
    row        = X_TEST_GLOBAL.iloc[idx].values.reshape(1, -1)
    churn_prob = float(model.predict_proba(row)[0][1])
    actual     = int(y_test.iloc[idx])

    state = {
        "customer_id":          f"cust_{idx:04d}",
        "customer_index":       idx,
        "customer_ltv":         1500.0,
        "churn_probability":    churn_prob,
        "risk_level":           "",
        "shap_reasons":         [],
        "raw_shap_values":      [],
        "action":               "",
        "planner_reasoning":    "",
        "decision_source":      "",
        "discount_info":        None,
        "crm_record":           None,
        "slack_log":            None,
        "execution_summary":    "",
        "customer_response":    "",
        "negotiation_round":    0,
        "conversation_history": [],
        "timestamps":           {}
    }

    final = app.invoke(state)

    results.append({
        "customer_id":      final["customer_id"],
        "actual_churn":     actual,
        "churn_prob":       round(churn_prob, 4),
        "risk_level":       final["risk_level"],
        "action":           final["action"],
        "decision_source":  final["decision_source"],
        "customer_response":final["customer_response"],
        "discount_%":       final["discount_info"]["discount_percentage"] if final["discount_info"] else None
    })


# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("batch_results.csv", index=False)

print("\n===== BATCH EVALUATION RESULTS =====")
print(results_df.to_string(index=False))


[Monitor] Evaluating churn risk...
[Monitor] Churn probability=0.82 → Risk level=high

[Explainer] Computing SHAP values...
[Explainer] Top reasons:
  - Total Charges = 5427.05 moderately increases churn risk (SHAP=1.1349999904632568)
  - Tenure Months = 57 slightly reduces churn risk (SHAP=-0.9580000042915344)
  - Dependents_Yes = False slightly increases churn risk (SHAP=0.34299999475479126)

[Planner] Calling LLM for retention decision...
[Planner] LLM decision → offer_discount | High confidence and moderately increasing total charges (SHAP=1.1349999904632568) justify offering a discount to mitigate churn risk.

[Executor] Executing action with tools...
[CRM] Customer cust_1309: last_action → offer_discount
[CRM] Customer cust_1309: churn_probability → 0.8235
[CRM] Customer cust_1309: risk_level → high
[Discount] 24.7% discount calculated for LTV=1500.0, risk=0.8234943747520447
[CRM] Customer cust_1309: discount_offered → 24.7
[Slack] sent: Churn Alert | Customer cust_1309 | Risk=H